# Нелинейные модели

## Импорт библиотек и загрузка данных

In [1]:
import numpy as np 
import pandas as pd 
import os

import matplotlib.pyplot as plt
import seaborn as sns

from bs4 import BeautifulSoup
import re
import string


from collections import Counter

import nltk
import spacy
from nltk import tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import Counter

from gensim.models import FastText
from gensim.models import KeyedVectors
from gensim.test.utils import common_texts
from gensim.models import Word2Vec
import gensim.downloader as api
import gensim

# Убедитесь, что необходимые ресурсы загружены
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('corpora')
nltk.download('wordnet')

 
nltk.download('stopwords')

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, MaxAbsScaler

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
import optuna

from sklearn.base import TransformerMixin, BaseEstimator


sns.set_theme(rc={'figure.figsize':(15, 7)})


import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE=42

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Error loading corpora: Package 'corpora' not found in
[nltk_data]     index
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
INPUT_DATA = "/kaggle/input/books-data/"

X_train = pd.read_csv(os.path.join(INPUT_DATA, 'X_train.csv'))
X_test = pd.read_csv(os.path.join(INPUT_DATA, 'X_test.csv'))
y_train = pd.read_csv(os.path.join(INPUT_DATA, 'y_train.csv'))
y_test = pd.read_csv(os.path.join(INPUT_DATA, 'y_test.csv'))

X_train.index = y_train.index
X_test.index = y_test.index

In [3]:
indx_train = list(y_train.query("author != 'William_Faulkner'").index)
indx_test = list(y_test.query("author != 'William_Faulkner'").index)

X_train, y_train = X_train.loc[indx_train, :], y_train.loc[indx_train, :]
X_test, y_test = X_test.loc[indx_test, :], y_test.loc[indx_test, :]

In [4]:
enc = LabelEncoder()

y_train = pd.Series(enc.fit_transform(y_train))
y_test = pd.Series(enc.transform(y_test))

In [5]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train)

X_train, X_valid, X_test = X_train['text'], X_valid['text'], X_test['text']

## LogisticRegression (model to compare)

Задачу классификации авторов отлично решают линейные модели - логистическая регрессия вместе с TfidfVectorizer отлично решают задачу без подбора гиперпараметров. Вот пример такой модели.

In [6]:
model = LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced') 
vector = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

# Создаем Pipeline
pipe = Pipeline([
    ('vectorizer', vector),
    ('scaler', MaxAbsScaler()),
    ('classifier', model)
])

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_valid)
y_pred_proba = pipe.predict_proba(X_valid)

score = f1_score(y_valid, y_pred, average='micro')

print('Метрика для линйеной модели:', score)

Метрика для линйеной модели: 0.8987341772151899


Если подобрать solver, `max_features` и т.п., то можно добиться более высоких результатов.

## Default Xgboost

In [7]:
class DenseTransformer(TransformerMixin):

    def fit(self, X, y=None, **fit_params):
        return self

    def transform(self, X, y=None, **fit_params):
        return X.todense()

Для начала стоит посмотреть, какое качество дает xgboost без какого-либо подбора гиперпараметров.

In [8]:
model = XGBClassifier(
                    random_state=RANDOM_STATE,
                    tree_method='gpu_hist',
                    device="cuda"
)


vector = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

# Создаем Pipeline
pipe = Pipeline([
    ('vectorizer', vector),
    ('dense', DenseTransformer()),
    ('classifier', model)
])

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_valid)
score = f1_score(y_valid, y_pred, average='micro')

print('Метрика для xgboost:', score)

Метрика для xgboost: 0.7341772151898734


Метрика значительно хуже, чем у логистической регрессии. Можно попробовать подобрать гиперпараметры, чтобы улучшить результат.

## CountVectorizer + XGBOOST

In [9]:
def objective(trial):
    # Определяем гиперпараметры для CountVectorizer
    max_features = trial.suggest_int('max_features', 10000, 50000)
    ngram_range = trial.suggest_int('ngram_range', 1, 3)
    
    # Определяем гиперпараметры для XGBoost
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),  
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),   
        'max_depth': trial.suggest_int('max_depth', 3, 50),   
        'subsample': trial.suggest_float('subsample', 0.5, 1),  
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1),  
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 25), 
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 25)
}
    

    model = XGBClassifier(
                        random_state=RANDOM_STATE,
                        tree_method='gpu_hist',
                        device="cuda"
    )
    
    vector = CountVectorizer(max_features=max_features, ngram_range=(1, ngram_range))


    # Создаем Pipeline
    pipe = Pipeline([
        ('vectorizer', vector),
        ('dense', DenseTransformer()),
        ('classifier', model)
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_valid)
    score = f1_score(y_valid, y_pred, average='micro')

    return score


# Создаем и запускаем оптимизацию
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

[I 2025-03-17 16:20:04,515] A new study created in memory with name: no-name-ee5106cf-3493-441c-80cf-b0b9f43c02ba
[I 2025-03-17 16:20:43,969] Trial 0 finished with value: 0.6962025316455697 and parameters: {'max_features': 43277, 'ngram_range': 1, 'n_estimators': 52, 'learning_rate': 0.1397908507280694, 'max_depth': 44, 'subsample': 0.7590871568799427, 'colsample_bytree': 0.5134534698838975, 'reg_alpha': 21.284098173312817, 'reg_lambda': 22.94405445821433}. Best is trial 0 with value: 0.6962025316455697.
[I 2025-03-17 16:21:15,684] Trial 1 finished with value: 0.6329113924050633 and parameters: {'max_features': 16329, 'ngram_range': 1, 'n_estimators': 70, 'learning_rate': 0.24741974038850137, 'max_depth': 33, 'subsample': 0.5793801728199635, 'colsample_bytree': 0.5123232533177706, 'reg_alpha': 22.361206728008426, 'reg_lambda': 2.870917676308319}. Best is trial 0 with value: 0.6962025316455697.
[I 2025-03-17 16:25:02,256] Trial 2 finished with value: 0.6835443037974683 and parameters: {

Лучшая метрика на валидации - $0.7$, что хуже чем вариант с tf-idf, но все же неплохо.

## `TF-IDF` и `Xgboost`

In [10]:
# Загрузка данных
def objective(trial):
    max_features = trial.suggest_int('max_features', 10000, 40000)
    ngram_range = trial.suggest_int('ngram_range', 1, 2)
    
    # Определяем гиперпараметры для XGBoost
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),  
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),   
        'max_depth': trial.suggest_int('max_depth', 3, 50),   
        'subsample': trial.suggest_float('subsample', 0.7, 1),  
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1),  
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 25), 
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 25)
}


    model = XGBClassifier(
                        random_state=RANDOM_STATE,
                        tree_method='gpu_hist',
                        device="cuda"
    )

    vector = TfidfVectorizer(max_features=max_features, ngram_range=(1, ngram_range))


    # Создаем Pipeline
    pipe = Pipeline([
        ('vectorizer', vector),
        ('dense', DenseTransformer()),
        ('classifier', model)
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_valid)
    score = f1_score(y_valid, y_pred, average='micro')

    return score

# Создаем и запускаем оптимизацию
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

[I 2025-03-17 16:33:22,551] A new study created in memory with name: no-name-859f1f9e-9385-452f-a0c1-344afcd6f1e3
[I 2025-03-17 16:35:06,990] Trial 0 finished with value: 0.6835443037974683 and parameters: {'max_features': 14673, 'ngram_range': 2, 'n_estimators': 70, 'learning_rate': 0.1198472952746331, 'max_depth': 7, 'subsample': 0.9786601174473217, 'colsample_bytree': 0.8081952051383505, 'reg_alpha': 24.63675050893598, 'reg_lambda': 17.25721248825792}. Best is trial 0 with value: 0.6835443037974683.
[I 2025-03-17 16:35:44,370] Trial 1 finished with value: 0.6835443037974683 and parameters: {'max_features': 15853, 'ngram_range': 1, 'n_estimators': 296, 'learning_rate': 0.142156283774088, 'max_depth': 28, 'subsample': 0.9329655900811833, 'colsample_bytree': 0.9636862477079204, 'reg_alpha': 15.603461800684345, 'reg_lambda': 5.654960065550882}. Best is trial 0 with value: 0.6835443037974683.
[I 2025-03-17 16:37:38,929] Trial 2 finished with value: 0.6708860759493671 and parameters: {'ma

Лучшая метрика на валидации - $0.759$, что является улучшением решения по умолчанию, но метрика все еще сильно хуже, чем у линейной модели. Проблема в том, что xgboost нехорошо справляется с этой задачей. Линейные модели без какого-либо тюнинга уже превосходят xgboost в качестве.

## `glove-wiki-gigaword-100`  + XGBoost

In [11]:
%%time

model = api.load("glove-wiki-gigaword-100")

# Функция для получения эмбеддингов
def get_embeddings(texts):
    embeddings = []
    for text in texts:
        # Получаем векторы для каждого слова в тексте
        word_vectors = [model[word] for word in text if word in model]
        # Если в тексте есть слова, получаем средний вектор
        if word_vectors:
            embeddings.append(np.mean(word_vectors, axis=0))
        else:
            embeddings.append(np.zeros(model.vector_size))  # Если нет слов, возвращаем нулевой вектор
    return embeddings

# Применение функции к колонке DataFrame
X_train_glove = pd.DataFrame(get_embeddings(X_train))
X_valid_glove = pd.DataFrame(get_embeddings(X_valid))
X_test_glove = pd.DataFrame(get_embeddings(X_test))

[==================================================] 100.0% 128.1/128.1MB downloaded
CPU times: user 6min 7s, sys: 18.4 s, total: 6min 26s
Wall time: 6min 26s


In [13]:
# Загрузка данных
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),  
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),   
        'max_depth': trial.suggest_int('max_depth', 3, 50),   
        'subsample': trial.suggest_float('subsample', 0.7, 1),  
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1),  
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 25), 
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 25)
}


    model = XGBClassifier(
                        **params,
                        random_state=RANDOM_STATE,
                        tree_method='gpu_hist',
                        device="cuda"
    )
   

    model.fit(X_train_glove, y_train)

    y_pred = model.predict(X_valid_glove)
    score = f1_score(y_valid, y_pred, average='micro')

    return score


# Создаем и запускаем оптимизацию
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

[I 2025-03-17 16:58:35,146] A new study created in memory with name: no-name-c9a40e0c-3f7a-4e90-8adb-3b3d95409650


The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


[I 2025-03-17 16:58:42,718] Trial 0 finished with value: 0.26582278481012656 and parameters: {'n_estimators': 278, 'learning_rate': 0.22479213889377342, 'max_depth': 8, 'subsample': 0.7521049479758514, 'colsample_bytree': 0.6975290005520491, 'reg_alpha': 8.807711536880266, 'reg_lambda': 10.34050376658336}. Best is trial 0 with value: 0.26582278481012656.
[I 2025-03-17 16:58:49,432] Trial 1 finished with value: 0.13924050632911392 and parameters: {'n_estimators': 245, 'learning_rate': 0.2914893174641276, 'max_depth': 16, 'subsample': 0.9313842464760569, 'colsample_bytree': 0.9563243321359893, 'reg_alpha': 14.355944418193596, 'reg_lambda': 11.022779879996083}. Best is trial 0 with value: 0.26582278481012656.
[I 2025-03-17 16:58:54,762] Trial 2 finished with value: 0.12658227848101267 and parameters: {'n_estimators': 196, 'learning_rate': 0.029066930493469832, 'max_depth': 42, 'subsample': 0.8333761839986834, 'colsample_bytree': 0.7558931325472735, 'reg_alpha': 13.024830299573738, 'reg_la

Метрика для DL-эмбеддингов сильно снизилась очень сильно (до $0.1$ - $0.4$). Почему-то эмбеддинги DL плохо решают задачу.

## `glove-wiki-gigaword-300`  + XGBoost

In [14]:
model = api.load("glove-wiki-gigaword-300")

# Функция для получения эмбеддингов
def get_embeddings(texts):
    embeddings = []
    for text in texts:
        # Получаем векторы для каждого слова в тексте
        word_vectors = [model[word] for word in text if word in model]
        # Если в тексте есть слова, получаем средний вектор
        if word_vectors:
            embeddings.append(np.mean(word_vectors, axis=0))
        else:
            embeddings.append(np.zeros(model.vector_size))  # Если нет слов, возвращаем нулевой вектор
    return embeddings

# Применение функции к колонке DataFrame
X_train_glove = pd.DataFrame(get_embeddings(X_train))
X_valid_glove = pd.DataFrame(get_embeddings(X_valid))
X_test_glove = pd.DataFrame(get_embeddings(X_test))

[===========================================-------] 87.3% 328.3/376.1MB downloaded

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=10000.0 (msgs/sec)
NotebookApp.rate_limit_window=1.0 (secs)



In [15]:
# Загрузка данных
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),  
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),   
        'max_depth': trial.suggest_int('max_depth', 3, 50),   
        'subsample': trial.suggest_float('subsample', 0.7, 1),  
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1),  
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 25), 
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 25)
}


    model = XGBClassifier(
                        random_state=RANDOM_STATE,
                        tree_method='gpu_hist',
                        device="cuda"
    )
   

    model.fit(X_train_glove, y_train)

    y_pred = model.predict(X_valid_glove)
    score = f1_score(y_valid, y_pred, average='micro')

    return score


# Создаем и запускаем оптимизацию
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

[I 2025-03-17 17:09:18,897] A new study created in memory with name: no-name-1fbdf4fc-ac89-4fbe-a0ec-e1f3a7d8c1cd
[I 2025-03-17 17:09:22,130] Trial 0 finished with value: 0.4810126582278481 and parameters: {'n_estimators': 272, 'learning_rate': 0.0367315003597167, 'max_depth': 29, 'subsample': 0.8079376340157933, 'colsample_bytree': 0.8915325593042057, 'reg_alpha': 4.161763289527202, 'reg_lambda': 18.48694960378555}. Best is trial 0 with value: 0.4810126582278481.
[I 2025-03-17 17:09:25,322] Trial 1 finished with value: 0.4810126582278481 and parameters: {'n_estimators': 267, 'learning_rate': 0.1867601739148527, 'max_depth': 9, 'subsample': 0.824139470677266, 'colsample_bytree': 0.9730859725355151, 'reg_alpha': 5.258008075009108, 'reg_lambda': 9.662116388324115}. Best is trial 0 with value: 0.4810126582278481.
[I 2025-03-17 17:09:28,508] Trial 2 finished with value: 0.4810126582278481 and parameters: {'n_estimators': 214, 'learning_rate': 0.20550868519087337, 'max_depth': 25, 'subsampl

Метрика на валидации примерно такая же, как и у предыдущего эмбеддинга.

## Тест

Теперу я выберу лучшую модель на валидации и протестирую её.

In [32]:
best_params = {'n_estimators': 69, 'learning_rate': 0.14121244292017354, 'max_depth': 28,
               'subsample': 0.8521996012524695, 'colsample_bytree': 0.714276105834954,  'reg_alpha': 2.9729140825120206,
               'reg_lambda': 15.122445446719771}

vector = TfidfVectorizer(max_features=17878, ngram_range=(1, 1))

model = XGBClassifier(
    **best_params,
    random_state=RANDOM_STATE,
    tree_method='gpu_hist',
    device="cuda"
)


# Создаем Pipeline
pipe = Pipeline([
    ('vectorizer', vector),
    ('dense', DenseTransformer()),
    ('classifier', model)
])


pipe.fit(X_train, y_train)


Pipeline(steps=[('vectorizer', TfidfVectorizer(max_features=17878)),
                ('dense', <__main__.DenseTransformer object at 0x79de5027e2f0>),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.714276105834954,
                               device='cuda', early_stopping_rounds=None,
                               enable_categorical=False, eval_...
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.14121244292017354, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=28,
                               max_leaves=None, min_child_weight=None,
                               missing=nan, monotone_constraints=None,
                               multi_strategy=None, n_estimators=69,
                               n_jobs=None, num_parallel_tree=None,
                               objective='multi:softprob', ...))])

In [33]:
y_pred_test = pipe.predict(X_test)
y_pred_proba_test = pipe.predict_proba(X_test)

In [37]:
f1 = f1_score(y_test, y_pred_test, average='micro')
accuracy = accuracy_score(y_test, y_pred_test)

print(
    f'f1_score={f1}',
    f'\naccuracy_score={accuracy}'
)

f1_score=0.5204081632653061 
accuracy_score=0.5204081632653061
